# 01 dashboard output contract

**Owner:** Thomas. Original notebook cells: 87, 88, 89.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

# 9. Dashboard Output Contract

The agents and model should not return arbitrary structures to the frontend. All MVP outputs should follow a stable contract so Thomas and Daniel can connect Databricks results to the Owner, Front Desk, and Optician dashboards without rewriting integration logic.

## Common output contract

```text
workflow_id
workflow_type
patient_or_record_id
staff_role
category
priority
status
summary
recommended_action
reason
source_system
source_timestamp
requires_human_review
model_score
model_version
retrieved_sources
agent_name
```

## Role-specific minimum fields

### Owner Dashboard
- unresolved high-priority item count
- insurance verification rate
- late optical-order count
- workflow completion metrics
- AI priority queue

### Front Desk Dashboard
- appointment readiness
- insurance verification status
- scheduling exception
- follow-up action
- priority / due time

### Optician Dashboard
- optical order status
- lab / vendor status
- days outstanding
- deposit / pickup status
- recommended follow-up

**Important:** The frontend should display only the minimum necessary information for the user's role. Sensitive or unrelated data should not be included simply because it exists in Crystal.


In [ ]:
# Shared dashboard output schema helper
DASHBOARD_OUTPUT_FIELDS = [
    "workflow_id",
    "workflow_type",
    "patient_or_record_id",
    "staff_role",
    "category",
    "priority",
    "status",
    "summary",
    "recommended_action",
    "reason",
    "source_system",
    "source_timestamp",
    "requires_human_review",
    "model_score",
    "model_version",
    "retrieved_sources",
    "agent_name",
]

def validate_dashboard_output(record: dict):
    missing = [field for field in DASHBOARD_OUTPUT_FIELDS if field not in record]
    return {
        "valid": len(missing) == 0,
        "missing_fields": missing,
    }


In [ ]:
# THOMAS: Structured dashboard output builder

from datetime import datetime, timezone

def build_dashboard_record(
    workflow_id,
    workflow_type,
    record_id,
    staff_role,
    category,
    priority,
    status,
    summary,
    recommended_action,
    reason,
    agent_name,
    requires_human_review=True,
    model_score=None,
    model_version=None,
    retrieved_sources=None,
    source_timestamp=None
):

    record = {
        "workflow_id": workflow_id,
        "workflow_type": workflow_type,
        "patient_or_record_id": str(record_id),
        "staff_role": staff_role,
        "category": category,
        "priority": priority,
        "status": status,
        "summary": summary,
        "recommended_action": recommended_action,
        "reason": reason,
        "source_system": "Synthetic Crystal-style data",
        "source_timestamp": source_timestamp,
        "requires_human_review": requires_human_review,
        "model_score": model_score,
        "model_version": model_version,
        "retrieved_sources": retrieved_sources or [],
        "agent_name": agent_name
    }

    validation = validate_dashboard_output(record)

    if not validation["valid"]:
        raise ValueError(
            validation["missing_fields"]
        )

    return record


def get_dashboard_records(staff_role, subject):

    records = []

    if staff_role not in ROLE_WORKFLOWS:
        raise AgentError("Unknown staff role.")

    if "Scheduling" in ROLE_WORKFLOWS[staff_role]:

        gw = CapstoneWorkflowGateway(
            staff_role, "Scheduling", subject
        )

        dataset = gw.fetch("GetNoShowRisk", {})

        for row in dataset.rows:

            records.append(build_dashboard_record(
                workflow_id="SCH-02",
                workflow_type="Scheduling",
                record_id=row["scheduleId"],
                staff_role=staff_role,
                category="No-show risk",
                priority=row["priority"],
                status=row["status"],
                summary="Synthetic appointment risk review",
                recommended_action=row[
                    "recommended_action"
                ],
                reason="Booking-time model risk score",
                agent_name="Scheduling Agent",
                requires_human_review=row[
                    "requires_human_review"
                ],
                model_score=row["no_show_score"],
                model_version=row["model_version"],
                source_timestamp=dataset.fetched_at
            ))

    if "Insurance Verification" in ROLE_WORKFLOWS[staff_role]:

        gw = CapstoneWorkflowGateway(
            staff_role, "Insurance Verification", subject
        )

        dataset = gw.fetch("GetVerificationStatus", {})

        for row in dataset.rows:

            flagged = (
                row["verification_status"] != "Verified"
            )

            records.append(build_dashboard_record(
                workflow_id=f"INS-{row['patient_id']}",
                workflow_type="Insurance Verification",
                record_id=row["patient_id"],
                staff_role=staff_role,
                category="Insurance readiness",
                priority="High" if flagged else "Routine",
                status=row["verification_status"],
                summary="Synthetic insurance review",
                recommended_action=(
                    "Review verification exception"
                    if flagged else "No follow-up required"
                ),
                reason=row["exception"] or "No exception",
                agent_name="Insurance Agent",
                requires_human_review=flagged,
                source_timestamp=dataset.fetched_at
            ))

    if "Optical Orders" in ROLE_WORKFLOWS[staff_role]:

        gw = CapstoneWorkflowGateway(
            staff_role, "Optical Orders", subject
        )

        dataset = gw.fetch(
            "GetOfficeFrameOrders",
            {
                "startDate": "2026-09-01",
                "endDate": "2026-09-30"
            }
        )

        for row in dataset.rows:

            status = row["status"]

            actions = {
                "InFulfillment": "Review fulfillment status",
                "ReadyForDispensing": "Review pickup readiness",
                "Dispensed": "No follow-up required"
            }

            record_id = (
                f"{row['patientAccountId']}-{row['id']}"
            )

            records.append(build_dashboard_record(
                workflow_id=f"OPT-{record_id}",
                workflow_type="Optical Orders",
                record_id=record_id,
                staff_role=staff_role,
                category="Order tracking",
                priority=(
                    "Routine" if status == "Dispensed"
                    else "Review"
                ),
                status=status,
                summary="Synthetic optical order review",
                recommended_action=actions.get(
                    status, "Review order status"
                ),
                reason="Current synthetic order status",
                agent_name="Optical Agent",
                requires_human_review=status != "Dispensed",
                source_timestamp=dataset.fetched_at
            ))

    return records


dashboard_records = get_dashboard_records(
    "Owner",
    "demo-owner"
)

display(pd.DataFrame(dashboard_records))

print("Dashboard records:", len(dashboard_records))

assert all(
    validate_dashboard_output(record)["valid"]
    for record in dashboard_records
)

optician_records = get_dashboard_records(
    "Optician",
    "demo-optician"
)

assert all(
    record["workflow_type"] == "Optical Orders"
    for record in optician_records
)

print("Dashboard contract and role checks passed.")